# 05 · dbt：做成品表（gold），带测试和说明书

## 这一章在整个项目里的位置

```
00–04 ✅ → [05 成品表 gold] → 06 自动化 → 07–09 训练模型、预测全蛋白组 → 10 对外 API
```

## 背景：为什么需要这一章

04 号做出来的 silver 是"**按来源**"整理的：ProteinGym 一张表，UniProt 一张表。但后面用的人要的是"**按用途**"整理的表：
- **07 号训练模型**：要每次测量的分数，加上蛋白的序列
- **09 号预测全蛋白组**：要全部 2 万个人类蛋白的清单
- **10 号对外查询**：要能用基因名查到蛋白

这一章用 **dbt** 做这些 **gold**（成品）表。写的时候还发现了一个会让模型**悄悄学错**的问题（第 3 步），也在这里解决。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
        ↓
dbt（装在本地）把每张表的 SQL 送给 Databricks 的 SQL 计算资源去执行
        ↓
Databricks 里建好 staging 视图和 gold 表
        ↓
你：在这里看结果，打开 dbt 的说明书网站，或者去 Databricks 网页的 Catalog 看
```

## 这一章做的 8 件事

| # | 做什么 | 用的技术 | 比喻 |
|---|---|---|---|
| 1 | 认识 dbt | dbt | 菜谱集：每张表一个配方 |
| 2 | 设计成品表 | 星型模型（维度表、事实表） | 测量记录表 + 样本信息表 |
| 3 | 把 DMS 的位置换算成 UniProt 编号 | 序列比对 | 两把刻度不同的尺子对齐 |
| 4 | 连上 Databricks | 一小时的临时令牌 | 一次性门禁卡 |
| 5 | 建表并测试 | `dbt build` | 按菜谱做菜，每道都质检 |
| 6 | 看成品表 | SQL | 尝菜 |
| 7 | 说明书网站和血缘图 | `dbt docs` | 整本菜谱书 |
| 8 | 让 CI 每次都检查 dbt | `dbt parse` | 菜谱有没有写错 |

**花费：0**。

---

## 准备

运行下面这一格：定义执行命令的小工具，连上 Databricks。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
from databricks.sdk import WorkspaceClient
from qafi_platform import lakehouse
from qafi_platform.lakehouse import sql
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
GOLD = "qafi_dev.gold"
print("Databricks:", w.config.host)

> 如果报错：在终端运行 `databricks auth login --profile qafi`，再运行这一格。

---

## 第 1 步 · dbt 是什么

### 在干什么
认识 dbt 的项目文件夹。

### 背景
04 号的 silver 是用 notebook 写的：要拆压缩包、解析文件，这些事 Python 加 Spark 最合适。gold 不一样：它完全是"**用几张表算出另一张表**"，全是 SQL。这类工作，公司里的标准工具是 **dbt**。

### 用了什么技术，为什么
**dbt**：每张表写成一个 SQL 文件，就是一句 `SELECT`，像一张**菜谱**。dbt 负责剩下的事：
- **按依赖顺序建表**：菜谱里写"用到了哪道半成品"（`ref('...')`），dbt 自己排出先做哪道
- **测试**：每道菜做完就质检
- **说明书和血缘图**：自动生成一本菜谱书（第 7 步）

项目在 [`dbt/`](../../dbt/) 文件夹里：

| 文件 | 比喻 | 内容 |
|---|---|---|
| [`dbt_project.yml`](../../dbt/dbt_project.yml) | 封面 | 项目名；staging 建成视图，marts 建成表 |
| [`profiles.yml`](../../dbt/profiles.yml) | 怎么进厨房 | 怎么连 Databricks。不存任何密码 |
| [`models/staging/`](../../dbt/models/staging/) | 备料区 | 每个来源轻度整理：改名、改类型。建成**视图**，放在 `staging` 楼层 |
| [`models/marts/`](../../dbt/models/marts/) | 成品区 | gold 表，放在 `gold` 楼层 |
| [`models/marts/_marts.yml`](../../dbt/models/marts/_marts.yml) | 菜谱说明和质检标准 | 每张表、每一列是什么意思，要通过哪些测试 |
| [`tests/`](../../dbt/tests/) | 特别的质检 | 写成 SQL 的自定义测试 |

**视图**（view）是什么：一句存起来的查询，不额外存一份数据，每次查它都现算。备料区用视图，省空间；成品区建成真正的表，查得快。

### 公司里什么情况怎么做
- **dbt 是做 SQL 转换的行业通用工具**：Databricks、Snowflake、BigQuery 都能用，换公司也用得上。写 dbt 模型的岗位叫 analytics engineer，数据工程师也常写
- **加一张新表**：在 `models/` 里加一个 SQL 文件，在 yml 里写上说明和测试，提交，走 Pull Request

### 面试怎么说
> "Gold tables are dbt models in git: each is a SELECT, dependencies are declared with ref(), and dbt builds them in order, runs the tests and generates documentation and lineage."

---

## 第 2 步 · 星型模型：维度表和事实表

### 在干什么
决定成品表怎么组织。

### 背景
成品表要回答"07、09、10 号的人拿来就能用"。公司里最常见的组织方式叫**星型模型**：一张记录"发生了什么"的大表，周围几张描述"是谁、是什么"的小表。

### 用了什么技术，为什么
| 表 | 类型 | 一行代表 | 比喻 | 谁用 |
|---|---|---|---|---|
| `fct_dms_measurement` | **事实表**（fact） | 一次测量：哪个实验、哪个突变、分数多少 | 测量记录表 | 07 号训练 |
| `dim_protein` | **维度表**（dimension） | 一个人类蛋白：编号、基因名、序列、有没有 DMS 数据 | 样本信息表 | 07 号取序列；09 号预测全部 20,652 个蛋白；10 号按基因名查 |
| `dim_dms_assay` | **维度表** | 一个 DMS 实验：测的哪个蛋白、测的什么性状、多少测量能换算位置 | 实验信息表 | 07 号挑选实验 |

事实表里只放编号和数字，描述信息放在维度表里，要用时用编号连起来。这样同一个蛋白的信息只存一份，改起来也只改一处。

### 公司里什么情况怎么做
- **事实表很大、维度表很小**：比如电商公司，每一笔订单是事实，顾客和商品是维度
- **名字前缀约定**：`fct_` 是事实表，`dim_` 是维度表，看名字就知道

### 面试怎么说
> "The gold layer is a star schema: fct_dms_measurement at the grain of one measurement, with dim_protein and dim_dms_assay describing the protein and the assay."

---

## 第 3 步 · 把 DMS 的位置换算成 UniProt 编号

### 在干什么
让每个测量的位置，都能对上 UniProt 标准序列的编号。

### 背景
写这一章时我查到：**人类 DMS 测量里，16% 的位置编号和 UniProt 标准序列对不上**。原因有两类：
- **只测了一小段**：很多实验只测一个结构域，比如 PIN1 那个实验只测了 39 个氨基酸，位置从这一小段的开头数。UniProt 是从整个蛋白的开头数
- **序列有一点点不同**：比如 KRAS 少一个氨基酸，整体错开一位

**为什么这很要紧**：AlphaFold 的结构，用的是 UniProt 的编号。07 号要从结构上取特征（比如这个位置埋在蛋白里面还是露在表面）。编号对不上，特征就会挂到**错的氨基酸**上，模型照样能训练，只是学到的是错的，而且没人会发现。

### 用了什么技术，为什么
比喻：两把刻度不同的尺子，先找到它们对齐的位置。按顺序试四种情况：

| 换算方式 | 什么情况 | 测量数 |
|---|---|---|
| `identical` | 实验的序列和 UniProt 一模一样，编号不用换 | 231,599 |
| `offset` | 实验测的是一小段：在整个蛋白里找到这一段，加上起点 | 38,973 |
| `kmer` | 序列有差别：拿突变前后共 15 个氨基酸，在整个蛋白里找**唯一**的位置 | 45,584 |
| `unmapped` | 以上都不行：留空，不硬猜 | 6,820（2.1%） |

**换算完 0 个错**：每个换算后的位置，都指向原来的那个氨基酸。

**两层测试保证它一直是对的**：
- **单元测试**：用一个自己编的小蛋白，四种情况各举一例，检查换算结果。这测的是**代码逻辑**，不依赖真实数据
- **数据测试**：真实数据里，每个换算后的位置都必须指向原来的氨基酸。这测的是**数据本身**

### 公司里什么情况怎么做
- **不同来源的编号、坐标、名字对不上**，是数据工程里最常见、也最危险的问题：用户编号、商品编号、基因坐标都一样。统一成一个标准，并且**写测试证明它对**
- **对不上的留空、标出来**，不猜。后面的人自己决定怎么处理

### 面试怎么说
> "16% of human DMS positions were numbered relative to the measured construct, not the UniProt sequence that AlphaFold uses. I converted them by exact match, fragment offset or a unique 15-residue window, left 2% unmapped rather than guess, and pinned it with a dbt unit test plus a data test that every converted position hits the right residue."

---

## 第 4 步 · 连上 Databricks

### 在干什么
让本地的 dbt 能在 Databricks 上执行 SQL。

### 背景
dbt 自己不算数据，它把 SQL 送给 Databricks 的 **SQL 计算资源**（warehouse）去执行。它需要三样东西：工作区地址、计算资源地址、一个通行令牌。

### 用了什么技术，为什么
**令牌用一小时的临时令牌**：从你已经登录好的 Databricks 命令行换出来（`databricks auth token`），一小时后自动作废。**不存任何长期密码**：和 `aws login`、02 号的 OIDC 是同一个原则。

### 执行

In [ ]:
os.environ.update(lakehouse.dbt_env(w))
run("cd dbt && uv run dbt debug --profiles-dir .")

### 结果怎么看
最后一行 `All checks passed!`，中间有一行 `Connection test: OK connection ok`。

> 过了一小时，dbt 报令牌过期：重新运行这一格，换一个新的。

### 公司里什么情况怎么做
- **正式环境里，dbt 不用人的令牌**：用一个专门给程序的"服务账号"（service principal），由 06 号的调度系统来跑

---

## 第 5 步 · 建表并测试：`dbt build`

### 在干什么
按依赖顺序，建好所有的表，同时跑所有测试。

### 执行
大约 2 分钟：

In [ ]:
run("cd dbt && uv run dbt build --profiles-dir .")

### 结果怎么看
最后一行 `Done. PASS=25 WARN=0 ERROR=0 SKIP=0`，一共 25 项（07c 又加了标签相关的模型和测试，所以你现在跑整个项目会看到 53 项）：

| 项目 | 数量 |
|---|---|
| staging 视图 | 4 |
| gold 表 | 3 |
| 单元测试（换算逻辑） | 1 |
| 数据测试：唯一、不为空、取值范围、表之间的关联 | 15 |
| 自定义测试：一个实验里一个突变只有一行；换算后的位置都对 | 2 |

**顺序是 dbt 自己排的**：先建 4 个视图，再跑单元测试，通过了才建事实表，事实表的测试通过了才建依赖它的两张维度表。**任何一项失败，依赖它的都会跳过**，坏数据不会往下流，和 04 号的质检是一个道理。

### 公司里什么情况怎么做
- **每个 Pull Request**：CI 在一个测试用的 catalog 上跑 `dbt build`，全过才能合并
- **正式环境**：06 号的调度系统按时跑 `dbt build`

### 面试怎么说
> "`dbt build` runs models and tests in dependency order, so a failing test stops everything downstream of it; unit tests check the SQL logic on hand-made inputs before the model is built."

---

## 第 6 步 · 看成品表

三张 gold 表各有多少行：

In [ ]:
sql(w, f'''
SELECT 'dim_protein' AS gold_table, count(*) AS rows FROM {GOLD}.dim_protein
UNION ALL SELECT 'dim_dms_assay', count(*) FROM {GOLD}.dim_dms_assay
UNION ALL SELECT 'fct_dms_measurement', count(*) FROM {GOLD}.fct_dms_measurement''')

位置是怎么换算的：

In [ ]:
sql(w, f"SELECT position_mapping, count(*) AS measurements FROM {GOLD}.fct_dms_measurement GROUP BY 1 ORDER BY 2 DESC")

一个具体例子：PIN1 那个只测了 39 个氨基酸的实验，它的第 1、2、3 位，在整个蛋白里是第几位：

In [ ]:
sql(w, f"SELECT dms_id, assay_position, uniprot_position, position_mapping, ref_aa, alt_aa, dms_score FROM {GOLD}.fct_dms_measurement WHERE dms_id LIKE 'PIN1%' AND alt_aa = 'A' ORDER BY assay_position LIMIT 3")

按基因名查一个蛋白（10 号对外查询就是这样查的）：

In [ ]:
sql(w, f"SELECT uniprot_id, entry_name, gene_name, gene_id, length, dms_assays, dms_measurements FROM {GOLD}.dim_protein WHERE gene_name = 'TP53'")

### 结果怎么看
- `dim_protein` 20,652 个蛋白；`dim_dms_assay` 94 个实验（96 个人类实验减去对不上编号的 Q53Z42 的 2 个）；`fct_dms_measurement` 32.3 万个测量
- PIN1：实验里的第 1、2、3 位，是整个蛋白的第 6、7、8 位，正好是 PIN1 序列上的 K、L、P
- TP53：编号 P04637，基因编号 7157，4 个 DMS 实验、23,449 个测量

---

## 第 7 步 · 说明书网站和血缘图

### 在干什么
看 dbt 自动生成的说明书：每张表、每一列是什么意思，有哪些测试，谁从谁来。

### 背景
表多了以后，最常被问的问题是"这一列是什么意思""这张表从哪来"。答案如果只在某个人的脑子里，这个人一走就没了。

### 用了什么技术，为什么
- **说明书**：`_marts.yml` 里写的说明，dbt 生成一个网站
- **说明同时写进了 Unity Catalog**：Databricks 网页上 Catalog 里也能看到同样的表说明和列说明
- **血缘图**：网站右下角的按钮。从 Databricks 的 silver、bronze，经过 staging，到三张 gold 表，一张图看清

### 执行
生成说明书，然后在你电脑上打开这个网站：

In [ ]:
run("cd dbt && uv run dbt docs generate --profiles-dir .")
docs = subprocess.Popen(["uv", "run", "dbt", "docs", "serve", "--profiles-dir", ".", "--port", "8081", "--no-browser"], cwd="dbt", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("在浏览器打开 http://localhost:8081")

### 怎么看
- 左边 **Projects → qafi → models → marts**：点 `fct_dms_measurement`，能看到说明、每一列的意思、它上面挂着的测试
- 右下角的 **蓝色按钮**（View Lineage Graph）：整张血缘图
- 再去 Databricks 网页：**Catalog → qafi_dev → gold → fct_dms_measurement**，Overview 里是同样的说明；**Lineage** 标签里是 Unity Catalog 自己记录的血缘

看完关掉这个网站：

In [ ]:
docs.terminate()

### 面试怎么说
> "Descriptions live next to the SQL in git, become a documentation site with a lineage graph, and are written into Unity Catalog as table and column comments, so the catalog explains itself."

---

## 第 8 步 · 让 CI 每次都检查 dbt

CI 里加了一步 `dbt parse`：每次推送，检查 dbt 项目有没有写错，比如引用了不存在的表、yml 格式错了。它**不连 Databricks**，所以在 GitHub 的机器上也能跑。

真正建表的 `dbt build` 要连 Databricks，公司里会给 CI 一个测试用的 catalog 和服务账号。我们的免费版先不做这一步。

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 05 dbt gold layer with tests and docs") && git push -q')

推送后去 Actions 页面看一眼：lint 里多了 `dbt parse` 那一步，应该是绿的。

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 dbt | SQL 模型 + `ref()` | 表的算法放在 Git 里，自动排顺序 |
| 2 星型模型 | 事实表 + 维度表 | 拿来就能用，信息只存一份 |
| 3 位置换算 | 序列比对 + 两层测试 | 不同来源的坐标统一，并且证明是对的 |
| 4 连接 | 一小时临时令牌 | 不存长期密码 |
| 5 建表 | `dbt build` | 测试失败，下游就不建 |
| 7 说明书 | `dbt docs` + Unity Catalog 注释 | 表自己会"说话" |
| 8 CI | `dbt parse` | 写错了马上知道 |

**一句话**：07 号要用的训练数据和蛋白清单都准备好了：成品表按用途组织，位置都换算成了 AlphaFold 用的编号，每一步都有测试。

## 面试题

1. **What is dbt, and why use it instead of plain SQL scripts?**
   > "dbt turns SELECT statements into tables in dependency order, and puts tests, documentation and lineage next to the SQL in git. Plain scripts need the order, the testing and the docs handled by hand."

2. **Facts and dimensions?**
   > "A fact table records events or measurements at a declared grain, mostly keys and numbers; dimension tables describe the entities those keys point to. Here: one row per DMS measurement, described by protein and assay dimensions."

3. **Data tests versus unit tests in dbt?**
   > "Data tests check the data that was built — uniqueness, nulls, relationships, custom SQL rules. Unit tests check the transformation logic on hand-made inputs, before the model runs on real data."

4. **Tell me about a data problem you caught before it reached a model.**
   > "16% of DMS measurements were numbered within the measured domain rather than the full protein. Structural features from AlphaFold use full-protein numbering, so training would silently have used features of the wrong residue. I converted the positions, left the uncertain 2% unmapped, and added tests so it can't regress."

**下一章 06 · 自动化**：让 04 号的 Databricks 任务和这一章的 dbt 按时自动跑，失败了自动通知。